# Retrieval Evaluation

Evaluate retrieval quality across three strategies: Dense, BM25, and Hybrid+Rerank.

In [1]:
import os
import sys
import json
from tqdm.auto import tqdm
from sqlalchemy.orm import Session

# Add project root to path
sys.path.append("../..")

from app.db.session import get_session_factory
from app.db.models import KnowledgeBase, ChunkMetadata, Document, ModelRegistry
from app.models_manager.registry import get_embedding_function
from app.rag.retriever import search_vectors, search_bm25, ensure_connection
from app.rag.pipeline import RAGPipeline
from app.rag.reranker import rerank

# ---- Configuration ----
# KNOWLEDGE_BASE_ID = "8825ed2f-fef5-47bb-ad46-8ed7f3caad95"
# KNOWLEDGE_BASE_ID = "0be5c50c-e103-493c-8e78-4fb58574be1d"
KNOWLEDGE_BASE_ID = "1a6b8438-8fec-43dd-884f-9adcfd979a79"

QA_FILE = "./results/qa_pairs.jsonl"
OUTPUT_FILE = "./results/retrieval_results.jsonl"
TOP_K = 5


## 1. Setup & Load Data

In [2]:
if not os.path.exists(QA_FILE):
    raise FileNotFoundError(f"{QA_FILE} not found. Run qa_generator.ipynb first.")

with open(QA_FILE, "r", encoding="utf-8") as f:
    qa_pairs = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(qa_pairs)} QA pairs.")

SessionLocal = get_session_factory()
db = SessionLocal()
ensure_connection()

kb = db.query(KnowledgeBase).filter(KnowledgeBase.id == KNOWLEDGE_BASE_ID).first()
if not kb:
    raise ValueError(f"Knowledge Base {KNOWLEDGE_BASE_ID} not found.")

embed_fn = get_embedding_function(db, kb.embedding_model_id)

pipeline = RAGPipeline(db)
reranker_model = db.query(ModelRegistry).filter(
    ModelRegistry.model_type == "reranker", ModelRegistry.is_default == True
).first()

def fetch_chunk_details(db_session, chunk_ids):
    chunks = db_session.query(ChunkMetadata, Document).join(Document, ChunkMetadata.document_id == Document.id).filter(ChunkMetadata.id.in_(chunk_ids)).all()
    res = {}
    for c, d in chunks:
        res[c.id] = {
            "chunk_id": c.id,
            "document_id": c.document_id,
            "document_title": d.title,
            "content": c.content
        }
    return res

def calculate_metrics(retrieved_chunk_ids, ground_truth_doc_id, chunk_details):
    # We consider it correct if ANY retrieved chunk belongs to the ground_truth document.
    retrieved_doc_ids = [chunk_details.get(cid, {}).get("document_id") for cid in retrieved_chunk_ids]
    
    correct_found = ground_truth_doc_id in retrieved_doc_ids
    correct_rank = None
    mrr = 0.0
    
    if correct_found:
        correct_rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
        mrr = 1.0 / correct_rank
        
    return correct_found, correct_rank, mrr


Loaded 100 QA pairs.


## 2. Evaluate Strategies

In [3]:
results = {
    "summary": {
        "dense": {"total_queries": 0, "correct_retrieved": 0, "accuracy@5": 0.0, "mrr": 0.0},
        "bm25": {"total_queries": 0, "correct_retrieved": 0, "accuracy@5": 0.0, "mrr": 0.0},
        "hybrid_rerank": {"total_queries": 0, "correct_retrieved": 0, "accuracy@5": 0.0, "mrr": 0.0}
    }
}

# Clear previous output file
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    pass

# Open the file in append mode once
with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
    for qa in tqdm(qa_pairs, desc="Evaluating Queries"):
        q = qa["question"]
        gt_doc_id = qa["doc_id"]
        
        # --- 1. Dense Only ---
        try:
            q_vec = embed_fn(q)
            dense_hits = search_vectors(kb.collection_name, q_vec, top_k=TOP_K)
            dense_chunk_ids = [h["id"] for h in dense_hits]
        except Exception as e:
            print(f"Dense search failed for query: {q}\n{e}")
            dense_chunk_ids = []
            
        dense_details = fetch_chunk_details(db, dense_chunk_ids)
        d_found, d_rank, d_mrr = calculate_metrics(dense_chunk_ids, gt_doc_id, dense_details)
        
        # --- 2. BM25 Only ---
        try:
            bm25_hits = search_bm25(db, q, top_k=TOP_K)
            bm25_chunk_ids = [h["id"] for h in bm25_hits]
        except Exception as e:
            print(f"BM25 search failed for query: {q}\n{e}")
            bm25_chunk_ids = []
            
        bm25_details = fetch_chunk_details(db, bm25_chunk_ids)
        b_found, b_rank, b_mrr = calculate_metrics(bm25_chunk_ids, gt_doc_id, bm25_details)
        
        # --- 3. Hybrid + Rerank ---
        try:
            hybrid_hits = pipeline.retrieve(q, [KNOWLEDGE_BASE_ID], top_k=60)
            if reranker_model and reranker_model.model_path:
                reranked = rerank(q, hybrid_hits, top_k=TOP_K, model_path=reranker_model.model_path)
            else:
                reranked = hybrid_hits[:TOP_K]
                
            hybrid_chunk_ids = [h.get("chunk_id") or h.get("id") for h in reranked]
            hybrid_details = { cid: h for cid, h in zip(hybrid_chunk_ids, reranked) }
        except Exception as e:
            print(f"Hybrid+Rerank failed for query: {q}\n{e}")
            hybrid_chunk_ids = []
            hybrid_details = {}
            
        h_found, h_rank, h_mrr = calculate_metrics(hybrid_chunk_ids, gt_doc_id, hybrid_details)
        
        # --- Incremental Save ---
        query_record = {
            "question": q,
            "ground_truth": {
                "document_source": qa["document_source"],
                "doc_id": gt_doc_id,
                "answer": qa["answer"]
            },
            "dense": {
                "metrics": {"correct_found": d_found, "correct_rank": d_rank, "mrr": d_mrr},
                "retrieved_chunks": dense_chunk_ids,
                "retrieved_details": [dense_details.get(cid, {}) for cid in dense_chunk_ids]
            },
            "bm25": {
                "metrics": {"correct_found": b_found, "correct_rank": b_rank, "mrr": b_mrr},
                "retrieved_chunks": bm25_chunk_ids,
                "retrieved_details": [bm25_details.get(cid, {}) for cid in bm25_chunk_ids]
            },
            "hybrid_rerank": {
                "metrics": {"correct_found": h_found, "correct_rank": h_rank, "mrr": h_mrr},
                "retrieved_chunks": hybrid_chunk_ids,
                "retrieved_details": [hybrid_details.get(cid, {}) for cid in hybrid_chunk_ids]
            }
        }
        
        f.write(json.dumps(query_record, ensure_ascii=False) + "\n")
        f.flush()

        # Update in-memory summary for printing later
        for strat, found, mrr in [("dense", d_found, d_mrr), ("bm25", b_found, b_mrr), ("hybrid_rerank", h_found, h_mrr)]:
            results["summary"][strat]["total_queries"] += 1
            if found:
                results["summary"][strat]["correct_retrieved"] += 1
            results["summary"][strat]["mrr"] += mrr


Evaluating Queries:   0%|          | 0/100 [00:00<?, ?it/s]